In [ ]:
from pkg import *

In [ ]:
#创建模型并计算水动力
#body = Create_geometry(x=30,y=30,z=4,h=0.9,m=30*30*1.1*1025,nx_distance=30.01,ny_distance=30.01,nx=10,ny=10,dof=6,mesh=4)
#6*6
body = Create_geometry(x=37.5,y=60,z=2,h=0.5,m=37.5*60*0.5*1025,nx_distance=37.51,ny_distance=0,nx=8,ny=1,dof=6,mesh=2)

In [ ]:
# 生成单个浮体
x,y,z = 300,60,2
mesh = 2
h = 0.5
body = cpt.RectangularParallelepiped(size=(x, y, z), resolution= (int(x/mesh),int(y/mesh),int(z/0.2)), center=(0, 0, h))
body.center_of_mass = (0,0,h)
body.mass = x*y*0.5*1025
body.add_all_rigid_body_dofs()
body.inertia_matrix = body.compute_rigid_body_inertia()
body.keep_immersed_part(free_surface=0)
body.rotation_center = body.center_of_mass
body.hydrostatic_stiffness = body.compute_hydrostatic_stiffness()
body.name = f"body_{body.mesh.nb_faces:04d}"
body.show_matplotlib()

In [ ]:
mm = body.inertia_matrix

In [ ]:
# 写出网格
filename = "OneRectang.gdf"
# cpt.io.mesh_writers.write_DAT(filename, body.mesh.vertices, body.mesh.faces)
cpt.io.mesh_writers.write_GDF(filename, body.mesh.vertices, body.mesh.faces)

In [ ]:
body.mesh.faces

In [ ]:
# 执行水动力计算
omega = np.linspace(0.1, 2, 40)
# omega = np.linspace(0.01, 8, 420)
bem_solver = cpt.BEMSolver()
problems = [cpt.RadiationProblem(body=body, omega=o,rho=1025,sea_bottom=-58.5,radiating_dof=dof) for o in omega for dof in body.dofs]
problems += [cpt.DiffractionProblem(omega=o, body=body,rho=1025,wave_direction=0,sea_bottom=-58.5) for o in omega]
result = bem_solver.solve_all(problems,n_jobs=20)
dataset = cpt.assemble_dataset(result,wavelength=True) # wavelength=True 时，计算波长

In [ ]:
# 执行水动力计算0.4157 , 0.4836,0.5754 ,0.715 ,1.0135
omega = [0.4836]
# omega = np.linspace(0.01, 8, 420)
bem_solver = cpt.BEMSolver()
problems = [cpt.RadiationProblem(body=body, omega=o,rho=1025,sea_bottom=-58.5,radiating_dof=dof) for o in omega for dof in body.dofs]
problems += [cpt.DiffractionProblem(omega=o, body=body,rho=1025,wave_direction=0,sea_bottom=-58.5) for o in omega]
result = bem_solver.solve_all(problems,n_jobs=2)
dataset = cpt.assemble_dataset(result,wavelength=True) # wavelength=True 时，计算波长

In [ ]:
cpt.io.xarray.separate_complex_values(dataset).to_netcdf(f'BM8_direaction0_full.nc',
            encoding={'radiating_dof': {'dtype': 'U'},
                        'influenced_dof': {'dtype': 'U'}})

In [ ]:
dataset

In [ ]:
cpt.io.xarray.separate_complex_values(dataset).to_netcdf(f'BM10_145_direaction180.nc',
            encoding={'radiating_dof': {'dtype': 'U'},
                        'influenced_dof': {'dtype': 'U'}})

In [ ]:
dataset = hydro(body,omega,depth=-58.5,wave_direction=0,HTM=False,save = False)

In [ ]:
cpt.io.xarray.separate_complex_values(dataset).to_netcdf(f'DMFEM66_5852_direction225_nosprace_m4.nc',
            encoding={'radiating_dof': {'dtype': 'U'},
                        'influenced_dof': {'dtype': 'U'}})

### 读取第三方网格进行capytaine进行数值计算

In [ ]:
# 读取第三方网格开展计算
mesh = cpt.io.mesh_loaders.load_mesh('E:\phd\Code\DM-FEM2D\hydro_output\sphere.dat', name=None)
body = cpt.FloatingBody(mesh=mesh, center_of_mass=(0,0,-2))
body.add_all_rigid_body_dofs()
body.keep_immersed_part()
body.show_matplotlib()

body.inertia_matrix = body.compute_rigid_body_inertia()
body.hydrostatic_stiffness = body.compute_hydrostatic_stiffness()
import pandas as pd
inertia_matrix_df = pd.DataFrame(body.inertia_matrix.values)
hydrostatic_stiffness_df = pd.DataFrame(body.hydrostatic_stiffness.values)
inertia_matrix_df.to_csv('inertia_matrix_values.csv', index=False)
hydrostatic_stiffness_df.to_csv('hydrostatic_stiffness_values.csv', index=False)


# omega = np.linspace(0.01, 8, 420)
# bem_solver = cpt.BEMSolver()
# problems = [cpt.RadiationProblem(body=body, omega=o,rho=1025,sea_bottom=-50,radiating_dof=dof) for o in omega for dof in body.dofs]
# problems += [cpt.DiffractionProblem(omega=o, body=body,rho=1025,wave_direction=0,sea_bottom=-50) for o in omega]
# result = bem_solver.solve_all(problems,n_jobs=20)
# cpt.io.xarray.separate_complex_values(dataset).to_netcdf(f'self_sphere.nc',
#             encoding={'radiating_dof': {'dtype': 'U'},
#                         'influenced_dof': {'dtype': 'U'}})



In [ ]:
# Read the CSV file into a Pandas DataFrame
inertia_matrix_df = pd.read_csv('hydrostatic_stiffness_values.csv')

inertia_matrix_df.values
